
 Import libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import math
import os

print("Libraries imported successfully")

Libraries imported successfully


 Load the dataset

In [3]:
df = pd.read_excel(
    r"C:\Users\grand\OneDrive\Desktop\-Delivery-Route-Delay-Forecasting\-Delivery-Route-Delay-Forecasting\dataset\amazon_delivery.csv.xlsx"
)

print("Dataset loaded successfully")

Dataset loaded successfully


Step 10: Understand the dataset

In [5]:
print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())


Shape: (43739, 16)

First 5 rows:


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [6]:
print("\nColumn names:")
print(df.columns.tolist())


Column names:
['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Order_Date', 'Order_Time', 'Pickup_Time', 'Weather', 'Traffic', 'Vehicle', 'Area', 'Delivery_Time', 'Category']


In [7]:

print("\nDataset information:")
df.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Order_ID         43739 non-null  object        
 1   Agent_Age        43739 non-null  int64         
 2   Agent_Rating     43685 non-null  float64       
 3   Store_Latitude   43739 non-null  float64       
 4   Store_Longitude  43739 non-null  float64       
 5   Drop_Latitude    43739 non-null  float64       
 6   Drop_Longitude   43739 non-null  float64       
 7   Order_Date       43739 non-null  datetime64[ns]
 8   Order_Time       43648 non-null  object        
 9   Pickup_Time      43739 non-null  object        
 10  Weather          43648 non-null  object        
 11  Traffic          43648 non-null  object        
 12  Vehicle          43739 non-null  object        
 13  Area             43739 non-null  object        
 14  Delivery_Time   

Step 11: Check missing values

In [8]:
print(df.isnull().sum())

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time         91
Pickup_Time         0
Weather            91
Traffic            91
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64


Step 12: Check duplicates

In [9]:
print("Duplicate rows:", df.duplicated().sum())

print("Duplicate Order IDs:",
      df["Order_ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Order IDs: 0


Step 13: Convert date and time columns

In [10]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

df["Order_Time"] = pd.to_timedelta(
    df["Order_Time"].astype(str),
    errors="coerce"
)

df["Pickup_Time"] = pd.to_timedelta(
    df["Pickup_Time"].astype(str),
    errors="coerce"
)

Step 14: Create time-based features

In [11]:
df["Order_Day"] = df["Order_Date"].dt.day

df["Order_Month"] = df["Order_Date"].dt.month

df["Order_Weekday"] = df["Order_Date"].dt.dayofweek

df["Order_Year"] = df["Order_Date"].dt.year

Step 15: Calculate Haversine distance

In [12]:
def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371

    lat1, lon1, lat2, lon2 = map(
        math.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return R * c

Apply the function:

In [13]:
df["Distance_KM"] = df.apply(
    lambda row: haversine_distance(
        row["Store_Latitude"],
        row["Store_Longitude"],
        row["Drop_Latitude"],
        row["Drop_Longitude"]
    ),
    axis=1
)

display(df[["Order_ID", "Distance_KM"]].head())

,Order_ID,Distance_KM
0,ialx566343618,3.025149
1,akqg208421122,20.183530
2,njpu434582536,1.552758
3,rjto796129700,7.790401
4,zguw716275638,6.210138


In [14]:
df["Delivery_Hours"] = (
    df["Delivery_Time"] / 60
)